In [1]:
import pandas as pd
import numpy as np

In [3]:
PATH_INMET = 'dados\clima\dados_A701_H_2025-01-01_2025-12-31.csv'

df = pd.read_csv(PATH_INMET, sep=';', skiprows=10, decimal=',', encoding='latin-1')

print('Shape:', df.shape)
print('Colunas:', list(df.columns))

Shape: (8760, 7)
Colunas: ['Data Medicao', 'Hora Medicao', 'PRECIPITACAO TOTAL, HORARIO(mm)', 'RADIACAO GLOBAL(Kj/mÂ²)', 'TEMPERATURA DO AR - BULBO SECO, HORARIA(Â°C)', 'UMIDADE RELATIVA DO AR, HORARIA(%)', 'Unnamed: 6']


<>:1: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:1: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\sergi\AppData\Local\Temp\ipykernel_18080\821058116.py:1: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  PATH_INMET = 'dados\clima\dados_A701_H_2025-01-01_2025-12-31.csv'


In [4]:
df = df.drop(columns=['Unnamed: 6'])

print('Colunas após limpeza:', list(df.columns))

Colunas após limpeza: ['Data Medicao', 'Hora Medicao', 'PRECIPITACAO TOTAL, HORARIO(mm)', 'RADIACAO GLOBAL(Kj/mÂ²)', 'TEMPERATURA DO AR - BULBO SECO, HORARIA(Â°C)', 'UMIDADE RELATIVA DO AR, HORARIA(%)']


In [5]:
df.columns = ['data', 'hora', 'precipitacao_mm', 'radiacao_kjm2', 'temp_ar', 'umidade_ar']

print('Novas colunas:', list(df.columns))

Novas colunas: ['data', 'hora', 'precipitacao_mm', 'radiacao_kjm2', 'temp_ar', 'umidade_ar']


In [6]:
df['hora'] = df['hora'].astype(str).str.zfill(4)

df['timestamp'] = pd.to_datetime(df['data'] + ' ' + df['hora'], format='%Y-%m-%d %H%M')

df = df[['timestamp', 'temp_ar', 'umidade_ar', 'radiacao_kjm2', 'precipitacao_mm']]

print(df['timestamp'].head(5))

0   2025-01-01 00:00:00
1   2025-01-01 01:00:00
2   2025-01-01 02:00:00
3   2025-01-01 03:00:00
4   2025-01-01 04:00:00
Name: timestamp, dtype: datetime64[us]


In [7]:
print('Nulos antes:')
print(df.isnull().sum())

df = df.interpolate(method='linear')

print('\nNulos depois:')
print(df.isnull().sum())

Nulos antes:
timestamp           0
temp_ar            46
umidade_ar         46
radiacao_kjm2      50
precipitacao_mm    46
dtype: int64

Nulos depois:
timestamp          0
temp_ar            0
umidade_ar         0
radiacao_kjm2      0
precipitacao_mm    0
dtype: int64


In [8]:
df['hora_int'] = df['timestamp'].dt.hour

df['tem_luz'] = (df['radiacao_kjm2'] > 50).astype(float)

luz_acumulada = []
acum = 0.0

for i in range(len(df)):
    if df['hora_int'].iloc[i] == 0:
        acum = 0.0
    acum += df['tem_luz'].iloc[i]
    luz_acumulada.append(round(acum, 1))

df['light_hours_day'] = luz_acumulada

df = df.drop(columns=['hora_int', 'tem_luz'])

print('Mínimo de horas de luz num dia:', df[df['timestamp'].dt.hour == 23]['light_hours_day'].min())
print('Máximo de horas de luz num dia:', df[df['timestamp'].dt.hour == 23]['light_hours_day'].max())

Mínimo de horas de luz num dia: 8.0
Máximo de horas de luz num dia: 18.0


In [9]:
np.random.seed(42)

chuva_6h = df['precipitacao_mm'].rolling(window=6, min_periods=1).sum()

soil_moisture = (
    68
    - (df['temp_ar'] - 22) * 1.2
    + chuva_6h * 0.8
    + np.random.normal(0, 4, len(df))
)

df['soil_moisture'] = np.clip(soil_moisture, 15, 100).round(1)

df = df.drop(columns=['radiacao_kjm2', 'precipitacao_mm'])

print('Estatísticas da umidade do solo:')
print(df['soil_moisture'].describe().round(2))

Estatísticas da umidade do solo:
count    8760.00
mean       70.60
std         7.63
min        43.10
25%        65.50
50%        70.70
75%        75.60
max       100.00
Name: soil_moisture, dtype: float64


In [10]:
df.to_csv('dados/inmet_limpo.csv', index=False)

print('Arquivo inmet_limpo.csv gerado com sucesso!')
print(f'\nShape final: {df.shape}')
print(f'Colunas: {list(df.columns)}')
print(f'\nPeríodo: {df["timestamp"].min()} → {df["timestamp"].max()}')
print('\nPrimeiras linhas:')
df.head()

Arquivo inmet_limpo.csv gerado com sucesso!

Shape final: (8760, 5)
Colunas: ['timestamp', 'temp_ar', 'umidade_ar', 'light_hours_day', 'soil_moisture']

Período: 2025-01-01 00:00:00 → 2025-12-31 23:00:00

Primeiras linhas:


,timestamp,temp_ar,umidade_ar,light_hours_day,soil_moisture
0,2025-01-01 00:00:00,21.2,76.0,0.0,70.9
1,2025-01-01 01:00:00,21.1,79.0,0.0,68.5
2,2025-01-01 02:00:00,20.5,82.0,0.0,72.4
3,2025-01-01 03:00:00,20.7,81.0,0.0,75.7
4,2025-01-01 04:00:00,20.2,83.0,0.0,69.2
